##Imports

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import DeltaTable
from pyspark.sql.window import Window

##Configuration

In [0]:
CATALOG = "workspace"
SCHEMA = "final"

BRONZE_TABLE = f"{CATALOG}.{SCHEMA}.bronze_customer"
SILVER_TABLE = f"{CATALOG}.{SCHEMA}.silver_customer"

In [0]:
customer_df = spark.table(BRONZE_TABLE)

display(customer_df)

##Record Count

In [0]:
print("Bronze Customer Count:")
print(customer_df.count())

##Schema Review

In [0]:
customer_df.printSchema()

##Data Profiling

In [0]:
from pyspark.sql.functions import col, sum

null_profile = customer_df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in customer_df.columns
])

display(null_profile)

##Data Cleansing Function

In [0]:
def clean_string(column_name):
    return regexp_replace(
                regexp_replace(
                    trim(col(column_name)),
                    "\\+",
                    ""
                ),
                "&",
                "and"
           )

##Clean Account Number

In [0]:
customer_clean_df = (
    customer_df
    .withColumn(
        "AccountNumber",
        clean_string("AccountNumber")
    )
)

##Type Standardization

In [0]:
customer_clean_df = (
    customer_clean_df
    .withColumn(
        "CustomerID",
        col("CustomerID").cast("long")
    )
    .withColumn(
        "PersonID",
        col("PersonID").cast("long")
    )
    .withColumn(
        "StoreID",
        col("StoreID").cast("long")
    )
    .withColumn(
        "TerritoryID",
        col("TerritoryID").cast("long")
    )
)

##ModifiedDate Conversion

In [0]:
customer_clean_df = (
    customer_clean_df
    .withColumn(
        "ModifiedDate",
        to_timestamp("ModifiedDate")
    )
)
display(
    customer_clean_df.select("ModifiedDate")
)


##Business Validation

In [0]:
customer_valid_df = customer_clean_df.filter(
    col("CustomerID").isNotNull()
)

In [0]:
customer_reject_df = customer_clean_df.filter(
    col("CustomerID").isNull()
)

##Rejected Count

In [0]:
print(
    f"Rejected Rows : {customer_reject_df.count()}"
)

##Deduplication

In [0]:
window_spec = Window.partitionBy(
    "CustomerID"
).orderBy(
    col("ingested_at").desc()
)

In [0]:
customer_dedup_df = (
    customer_valid_df
    .withColumn(
        "rn",
        row_number().over(window_spec)
    )
    .filter(col("rn") == 1)
    .drop("rn")
)

##DQ Check 1

In [0]:
dq_customerid = customer_dedup_df.filter(
    col("CustomerID").isNull()
).count()

print(
    f"CustomerID Nulls : {dq_customerid}"
)

##DQ Check 2

In [0]:
dq_account = customer_dedup_df.filter(
    col("AccountNumber").isNull()
).count()

print(
    f"AccountNumber Nulls : {dq_account}"
)

##DQ Check 3

In [0]:
dq_duplicate = (
    customer_dedup_df
    .groupBy("CustomerID")
    .count()
    .filter(col("count") > 1)
    .count()
)

print(
    f"Duplicate Customers : {dq_duplicate}"
)

##Add Silver Metadata

In [0]:
silver_customer_df = (
    customer_dedup_df
    .withColumn(
        "silver_load_timestamp",
        current_timestamp()
    )
    .withColumn(
        "silver_layer",
        lit("SILVER")
    )
)

##Write Silver

In [0]:
(
    silver_customer_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema","true")
    .saveAsTable(SILVER_TABLE)
)

##Validation

In [0]:
silver_df = spark.table(SILVER_TABLE)

print(
    f"Silver Count : {silver_df.count()}"
)

display(silver_df)